|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Static batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: pick the batch size that is actually fastest<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(2)

You have measured how much faster a big batch is. Now find the batch size
that is actually fastest, which is not the same question.

All simulation, no GPU. The throughput numbers come from the demo notebook in
this folder.

In [2]:
### run this cell

lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=20000).astype(int) + 1

# The one hardware number in this notebook. B_ridge is the batch at which a
# decode step stops being free. Below it, extra sequences travel on a weight
# read that you already perform.
#
# This is Part 1's roofline, in sequences instead of FLOP per byte. Measure
# your own value in part2_pad_freeSequences. Then change this line.
B_RIDGE = 32

Bs = np.array([1,2,4,8,16,32,64,128,256])
print(f'{len(lengths)} requests, median {np.median(lengths):.0f} tokens, '
      f'p99 {np.percentile(lengths,99):.0f}')
print(f'B_ridge = {B_RIDGE}')

20000 requests, median 122 tokens, p99 980
B_ridge = 32


# Exercise 1: how much does each batch size waste?

A static batch of B occupies B slots for `max(lengths)` steps. Useful work is
`sum(lengths)`. The rest is a finished sequence holding a slot open.

In [3]:
def waste(batch):
  return 1 - batch.sum() / (batch.max() * len(batch))

def mean_waste(B, trials=500):
  return np.mean([waste(rng.choice(lengths, B)) for _ in range(trials)])

w = np.array([mean_waste(int(B)) for B in Bs])
for B, x in zip(Bs, w):
  print(f'batch {B:>4}: {100*x:5.1f}% wasted')

batch    1:   0.0% wasted
batch    2:  26.6% wasted
batch    4:  46.1% wasted
batch    8:  59.1% wasted
batch   16:  68.9% wasted
batch   32:  76.0% wasted
batch   64:  80.6% wasted
batch  128:  84.8% wasted
batch  256:  87.5% wasted


# Exercise 2: what you offered against what you delivered

Multiply the measured speedup by the fraction of slots that did real work,
and find the peak.

In [4]:
raw = np.minimum(Bs, B_RIDGE)          # min(B, B_ridge): the roofline

# what you actually get is what the hardware offered, minus the padding
delivered = raw * (1 - w)

best = Bs[np.argmax(delivered)]
print(f"{'batch':>6} {'offered':>8} {'wasted':>8} {'delivered':>10} {'kept':>6}")
for B, r, x, d in zip(Bs, raw, w, delivered):
  mark = '  <-- best' if B == best else ''
  print(f'{B:>6} {r:>7.0f}x {100*x:>7.1f}% {d:>9.1f}x {100*d/r:>5.0f}%{mark}')

 batch  offered   wasted  delivered   kept
     1       1x     0.0%       1.0x   100%
     2       2x    26.6%       1.5x    73%
     4       4x    46.1%       2.2x    54%
     8       8x    59.1%       3.3x    41%
    16      16x    68.9%       5.0x    31%
    32      32x    76.0%       7.7x    24%  <-- best
    64      32x    80.6%       6.2x    19%
   128      32x    84.8%       4.9x    15%
   256      32x    87.5%       4.0x    12%


# Exercise 3: can you cheat the distribution?

The tail sets the length of the batch. So keep the tail out of the batch:
gather a larger pool, sort it by length, and cut it into batches of similar
requests.

In [5]:
def bucketed_waste(B, n_buckets, trials=500):
  out = []
  for _ in range(trials):
    pool = rng.choice(lengths, B*n_buckets)
    pool.sort()                                  # group similar lengths
    out.append(np.mean([waste(pool[i*B:(i+1)*B]) for i in range(n_buckets)]))
  return np.mean(out)

B = int(best)
print(f'batch {B}, unsorted:            {100*mean_waste(B):5.1f}% wasted')
for nb in (2, 4, 8):
  print(f'batch {B}, sorted into {nb:>2} buckets: {100*bucketed_waste(B, nb):5.1f}% wasted')

batch 32, unsorted:             75.3% wasted
batch 32, sorted into  2 buckets:  56.2% wasted
batch 32, sorted into  4 buckets:  37.2% wasted
batch 32, sorted into  8 buckets:  22.5% wasted


# Exercise 4: change the machine, leave the traffic alone

Everything above used one value of `B_ridge`. Sweep it. For each machine,
report the best batch size and the fraction of the offered speedup that
survives padding.

In [6]:
print(f"{'B_ridge':>8} {'best batch':>11} {'delivered':>10} {'kept':>6}")
for br in (8, 16, 32, 64, 128, 256):
  d = np.minimum(Bs, br) * (1 - w)
  i = int(np.argmax(d))
  print(f'{br:>8} {Bs[i]:>11} {d[i]:>9.1f}x {100*d[i]/br:>5.0f}%')

 B_ridge  best batch  delivered   kept
       8           8       3.3x    41%
      16          16       5.0x    31%
      32          32       7.7x    24%
      64          64      12.4x    19%
     128         128      19.4x    15%
     256         256      32.0x    12%


### What survives a change of hardware

**The best batch size is `B_ridge`.** Do not memorise a number. It is the batch
at which your card stops giving sequences away free. It is the only property
of the machine in this notebook.

**And the fraction you keep falls as `B_ridge` rises.** A faster card permits a
larger batch. A larger batch holds a long outlier more often. The outlier sets
the length of the batch. So padding takes a larger share of a better machine.
That is the opposite of what you want.

So carry away a formula and a direction, not a measurement:

> delivered = `min(B, B_ridge)` x `(1 - waste(B))`

The first factor is your hardware. The second is your traffic. Better hardware
gives less and less until you fix the second factor.

**A sort by length helps a lot, and you cannot do it.** `pool.sort()` used the
output length. That is the number of tokens that the model has not made yet.
You do know the *prompt* length, and you could sort by that. But prompt length
hardly predicts output length.

So one obvious fix needs an oracle, and the other uses a weak substitute. The
fix that works needs neither. It stops the engine from forming batches in
advance.

    ./vc guide 5